In [18]:
import os, json, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

# ---- device ----
DEVICE = 'cpu'
if torch.cuda.is_available():
    try:
        _p = nn.Conv2d(1, 1, 3).cuda()
        _ = _p(torch.zeros(1, 1, 8, 8, device='cuda')).cpu()
        del _p
        DEVICE = 'cuda'
    except Exception as e:
        print(f'GPU unusable: {str(e)[:80]} -> CPU')
print(f'Device: {DEVICE}')

# ---- paths ----
COMP_ROOT  = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR   = Path(COMP_ROOT) / 'test'
OUT_DIR    = Path('/kaggle/working')
MODELS_DIR = Path('/kaggle/input/models')

print(f'TEST_DIR exists: {TEST_DIR.exists()}')
print(f'MODELS_DIR contents:')
for p in sorted(MODELS_DIR.rglob('*')):
    print(f'  {p}')

# ---- load metadata ----
meta_path = next(MODELS_DIR.rglob('*.json'), None)
if meta_path is None:
    raise FileNotFoundError(f'No metadata JSON found under {MODELS_DIR}')
with open(meta_path) as f:
    saved_meta = json.load(f)
print(f'\nMetadata: {saved_meta}')

BASE         = int(saved_meta['base'])
POOL         = int(saved_meta['pool'])
N_CONTEXT    = int(saved_meta.get('n_context', 3))
GAUSS_SIGMA  = float(saved_meta.get('gauss_sigma', 2.0))
CLAHE_CLIP   = float(saved_meta.get('clahe_clip', 3.0))
CLAHE_GRID   = int(saved_meta.get('clahe_grid', 8))
BEST_THRESH  = float(saved_meta.get('best_thresh', 0.20))
MIN_PEAK_DISTANCE = int(saved_meta.get('min_peak_distance', 9))

SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X um/voxel

print(f'\nBase={BASE} | Pool={POOL} | N_context={N_CONTEXT}')
print(f'CLAHE clip={CLAHE_CLIP} | Best thresh={BEST_THRESH}')
print(f'MIN_PEAK_DISTANCE={MIN_PEAK_DISTANCE}')

# ---- load weights ----
ckpt_path = next(MODELS_DIR.rglob('unet2d_best.pt'),
            next(MODELS_DIR.rglob('*.pt'), None))
if ckpt_path is None:
    raise FileNotFoundError(f'No .pt checkpoint found under {MODELS_DIR}')
print(f'\nCheckpoint: {ckpt_path}')
raw_state_dict = torch.load(ckpt_path, map_location='cpu')
print(f'State dict: {len(raw_state_dict)} tensors')

# infer BASE from weight if metadata didn't have it
_inferred_base = int(raw_state_dict['e1.0.weight'].shape[0])
if _inferred_base != BASE:
    print(f'WARNING: inferred BASE={_inferred_base} from weights, '
          f'overriding metadata BASE={BASE}')
    BASE = _inferred_base

Device: cpu
TEST_DIR exists: True
MODELS_DIR contents:
  /kaggle/input/models/sumantvj
  /kaggle/input/models/sumantvj/biohub-cell-detection
  /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch
  /kaggle/input/models/sumantvj/biohub-cell-detection/pytorch/default
  /kaggle/input/models/sumantvj/biohub-more-training-data
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1/training_history.csv
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1/unet2d_best.pt
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1/unet2d_final.pt
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1/unet2d_meta.json
  /kaggle/input/models/sumantvj/biohub-more-training-data/pytorch/default/1/vis_

In [19]:
def _block2d(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )

class UNet2D(nn.Module):
    """Must match training architecture exactly."""
    def __init__(self, base=BASE, n_context=N_CONTEXT):
        super().__init__()
        b = base
        self.e1   = _block2d(n_context, b)
        self.e2   = _block2d(b,    b*2)
        self.e3   = _block2d(b*2,  b*4)
        self.pool = nn.MaxPool2d(2)
        self.bn   = _block2d(b*4,  b*8)
        self.u3   = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.d3   = _block2d(b*8,  b*4)
        self.u2   = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.d2   = _block2d(b*4,  b*2)
        self.u1   = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.d1   = _block2d(b*2,  b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b  = self.bn(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet2D(base=BASE, n_context=N_CONTEXT).to(DEVICE)
model.load_state_dict(raw_state_dict, strict=True)
model.eval()
print(f'UNet2D loaded: {sum(p.numel() for p in model.parameters()):,} params')

with torch.no_grad():
    _t = torch.zeros(1, N_CONTEXT, 64, 64, device=DEVICE)
    _o = model(_t)
    assert _o.shape == (1, 1, 64, 64)
    print(f'Forward OK: {tuple(_t.shape)} -> {tuple(_o.shape)}')
    del _t, _o

UNet2D loaded: 1,085,737 params
Forward OK: (1, 3, 64, 64) -> (1, 1, 64, 64)


In [20]:
_ZC = {}

def read_meta(zp):
    with open(Path(zp) / '0' / 'zarr.json') as f:
        m = json.load(f)
    return dict(shape=tuple(m['shape']), dtype=np.dtype(m['data_type']))

def load_vol(zp, t, meta=None):
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode='r')['0']
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_meta(zp)
        buf = blosc2.decompress(
            open(Path(zp)/'0'/'c'/str(t)/'0'/'0'/'0', 'rb').read()
        )
        return np.frombuffer(buf, dtype=meta['dtype']).reshape(meta['shape'][1:])

def pool_xy(vol, f=POOL):
    Z, Y, X = vol.shape
    Y2, X2 = (Y//f)*f, (X//f)*f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2//f, f, X2//f, f).mean(axis=(2, 4))

def normalize_slice_clahe(slc, clip=CLAHE_CLIP, grid=CLAHE_GRID):
    """Matches training preprocessing exactly -- same clip and grid."""
    lo = float(np.percentile(slc, 2.0))
    hi = float(np.percentile(slc, 99.0))
    if hi <= lo:
        return np.zeros_like(slc, dtype=np.float32)
    scaled = np.clip((slc - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
    clahe  = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(scaled).astype(np.float32) / 255.0

def load_slice_context(pvol, z, n_context=N_CONTEXT):
    Z   = pvol.shape[0]
    rad = n_context // 2
    slices = []
    for dz in range(-rad, rad + 1):
        zi = int(np.clip(z + dz, 0, Z - 1))
        slices.append(normalize_slice_clahe(pvol[zi]))
    return np.stack(slices, axis=0)  # (N_CONTEXT, H, W)

print(f'I/O helpers ready. CLAHE clip={CLAHE_CLIP} grid={CLAHE_GRID}')

I/O helpers ready. CLAHE clip=3.0 grid=8


In [21]:
NMS_RADIUS_XY  = 3.0
NMS_RADIUS_Z   = 2
MIN_CONSEC_Z   = 2

def adaptive_threshold(pvol, base_thresh=BEST_THRESH):
    """
    Samples 8 slices to check the model's actual activation range.
    If the volume is dimmer than usual, lowers threshold proportionally.
    This fixed zero-detection on 44b6_0b24845f which had heatmap max ~0.30.
    """
    sample_zs = list(range(0, pvol.shape[0], max(1, pvol.shape[0]//8)))
    peak_vals = []
    with torch.no_grad():
        for z in sample_zs:
            ctx = load_slice_context(pvol, z)
            xt  = torch.from_numpy(ctx[None]).to(DEVICE)
            hm  = torch.sigmoid(model(xt))[0, 0].cpu().numpy()
            peak_vals.append(float(hm.max()))
    vol_max = float(np.median(peak_vals))
    if vol_max < base_thresh * 1.5:
        adaptive = max(0.05, vol_max * 0.5)
        print(f'    [adaptive] vol_max={vol_max:.3f} -> thresh={adaptive:.3f}')
        return adaptive
    return base_thresh

def detect_volume(pvol, thresh=None):
    """
    Runs 2.5D UNet slice-by-slice.
    Returns (confirmed, scores) where confirmed is list of (z, py, px)
    in pooled voxel coords, scores is list of float heatmap confidence.
    """
    if thresh is None:
        thresh = adaptive_threshold(pvol)

    model.eval()
    Z = pvol.shape[0]
    slice_dets   = {}
    slice_scores = {}

    with torch.no_grad():
        for z in range(Z):
            ctx = load_slice_context(pvol, z)
            xt  = torch.from_numpy(ctx[None]).to(DEVICE)
            hm  = torch.sigmoid(model(xt))[0, 0].cpu().numpy()
            pk  = peak_local_max(hm, min_distance=MIN_PEAK_DISTANCE,
                                  threshold_abs=thresh, exclude_border=False)
            slice_dets[z]   = pk
            slice_scores[z] = np.array([hm[p[0], p[1]] for p in pk]) \
                               if len(pk) > 0 else np.array([])

    confirmed = []
    scores    = []
    for z in range(Z):
        pk_z  = slice_dets.get(z,  np.zeros((0, 2)))
        scr_z = slice_scores.get(z, np.array([]))
        if len(pk_z) == 0:
            continue
        for idx, det_yx in enumerate(pk_z):
            consec = 1
            cur_yx = det_yx.copy()
            for dz in range(1, MIN_CONSEC_Z):
                pk_next = slice_dets.get(z + dz, np.zeros((0, 2)))
                if len(pk_next) == 0:
                    break
                dists = np.linalg.norm(pk_next - cur_yx, axis=1)
                ci    = np.argmin(dists)
                if dists[ci] <= NMS_RADIUS_XY * 2:
                    consec += 1
                    cur_yx  = pk_next[ci]
                else:
                    break
            if consec >= MIN_CONSEC_Z:
                confirmed.append((z, float(det_yx[0]), float(det_yx[1])))
                scores.append(float(scr_z[idx]) if idx < len(scr_z) else 0.0)

    return confirmed, scores

def nms_3d_weighted(confirmed, scores,
                     radius_xy=NMS_RADIUS_XY, radius_z=NMS_RADIUS_Z):
    """
    3D NMS with score-weighted centroid.
    Returns (centroids, nms_scores) -- both lists, same length.
    Score-weighted centroid gives sub-voxel precision vs peak-position snap.
    """
    if len(confirmed) == 0:
        return [], []
    dets  = np.array(confirmed, dtype=np.float64)
    scrs  = np.array(scores,    dtype=np.float64)
    order = np.argsort(scrs)[::-1]
    keep_centroids = []
    keep_scores    = []
    suppressed     = np.zeros(len(dets), dtype=bool)

    for i in order:
        if suppressed[i]:
            continue
        zi, yi, xi = dets[i]
        cluster = []
        for j in range(len(dets)):
            if suppressed[j]:
                continue
            zj, yj, xj = dets[j]
            if abs(zi-zj) <= radius_z and \
               np.sqrt((yi-yj)**2 + (xi-xj)**2) <= radius_xy:
                cluster.append(j)
                suppressed[j] = True
        cd = dets[cluster]; cs = scrs[cluster]
        tw = cs.sum()
        if tw > 0:
            z_w = float(np.dot(cs, cd[:,0]) / tw)
            y_w = float(np.dot(cs, cd[:,1]) / tw)
            x_w = float(np.dot(cs, cd[:,2]) / tw)
        else:
            z_w, y_w, x_w = float(zi), float(yi), float(xi)
        keep_centroids.append((z_w, y_w, x_w))
        keep_scores.append(float(cs.max()))

    return keep_centroids, keep_scores

print('Detection functions ready.')
print(f'BEST_THRESH={BEST_THRESH} | NMS_RADIUS_XY={NMS_RADIUS_XY} | '
      f'NMS_RADIUS_Z={NMS_RADIUS_Z} | MIN_CONSEC_Z={MIN_CONSEC_Z}')

Detection functions ready.
BEST_THRESH=0.05 | NMS_RADIUS_XY=3.0 | NMS_RADIUS_Z=2 | MIN_CONSEC_Z=2


In [22]:
MAX_LINK_UM   = 15.0
DIV_MAX_UM    = 8.0
MIN_DAUGHT_UM = 3.0

def link_timepoints(prev_nodes, curr_nodes):
    """
    Two-stage linker:
    Stage 1: 1-to-1 Hungarian on motion-predicted positions + confidence penalty.
    Stage 2: division check with tighter distance gate + daughter separation check.
    prev_nodes / curr_nodes: list of dicts with keys
        id, phys_z, phys_y, phys_x, score, parent_phys (optional)
    """
    if not prev_nodes or not curr_nodes:
        return []

    pos0 = np.array([[n['phys_z'], n['phys_y'], n['phys_x']] for n in prev_nodes])
    pos1 = np.array([[n['phys_z'], n['phys_y'], n['phys_x']] for n in curr_nodes])

    # motion-predicted positions for prev nodes
    pred0 = []
    for i, n in enumerate(prev_nodes):
        if n.get('parent_phys') is not None:
            vel = np.clip(pos0[i] - n['parent_phys'], -6.0, 6.0)
            pred0.append(pos0[i] + vel)
        else:
            pred0.append(pos0[i])
    pred0 = np.array(pred0)

    dist = np.sqrt(((pred0[:, None] - pos1[None, :])**2).sum(2))
    scrs1 = np.array([n['score'] for n in curr_nodes])
    cost  = dist + 1.5 * (1.0 - scrs1)[None, :]
    cost[dist > MAX_LINK_UM] = 1e9

    row_ind, col_ind = linear_sum_assignment(cost)
    links        = []
    matched_curr = set()
    prev_to_curr = {}

    for r, c in zip(row_ind, col_ind):
        if dist[r, c] <= MAX_LINK_UM:
            links.append({
                'source_id': prev_nodes[r]['id'],
                'target_id': curr_nodes[c]['id'],
                'source_phys': pos0[r],
            })
            matched_curr.add(c)
            prev_to_curr.setdefault(r, []).append(c)

    # division check
    for ci, n1 in enumerate(curr_nodes):
        if ci in matched_curr:
            continue
        best_ri, best_d = None, DIV_MAX_UM
        for ri in range(len(prev_nodes)):
            if ri not in prev_to_curr or len(prev_to_curr[ri]) != 1:
                continue
            if dist[ri, ci] > DIV_MAX_UM:
                continue
            exist_ci = prev_to_curr[ri][0]
            daught_sep = np.linalg.norm(pos1[ci] - pos1[exist_ci])
            if daught_sep < MIN_DAUGHT_UM:
                continue
            if dist[ri, ci] < best_d:
                best_d  = dist[ri, ci]
                best_ri = ri
        if best_ri is not None:
            links.append({
                'source_id': prev_nodes[best_ri]['id'],
                'target_id': curr_nodes[ci]['id'],
                'source_phys': pos0[best_ri],
            })
            prev_to_curr[best_ri].append(ci)

    return links

print(f'Linker ready. MAX_LINK={MAX_LINK_UM}um | DIV_MAX={DIV_MAX_UM}um | '
      f'MIN_DAUGHT_SEP={MIN_DAUGHT_UM}um')

Linker ready. MAX_LINK=15.0um | DIV_MAX=8.0um | MIN_DAUGHT_SEP=3.0um


In [ ]:
import networkx as nx

MIN_TRACK_LEN = 3

test_names = sorted(
    d.replace('.zarr', '')
    for d in os.listdir(TEST_DIR)
    if d.endswith('.zarr')
)
print(f'Test datasets: {len(test_names)}')
for n in test_names:
    print(f'  {n}')

all_rows = []

for folder_name in test_names:
    zp   = str(TEST_DIR / f'{folder_name}.zarr')
    meta = read_meta(zp)
    n_t  = meta['shape'][0]
    print(f'\n{folder_name} | {n_t} timepoints | shape {meta["shape"]}')

    all_nodes_by_t = {}
    edges_list     = []
    global_nid     = 0   # resets per dataset

    for t in range(n_t):
        vol  = load_vol(zp, t, meta)
        pvol = pool_xy(vol)

        confirmed, scores = detect_volume(pvol)
        nuclei, nms_scores = nms_3d_weighted(confirmed, scores)

        nodes_t = []
        for i, (z, py, px) in enumerate(nuclei):
            orig_y = py * POOL
            orig_x = px * POOL
            nodes_t.append({
                'id':          global_nid,
                't':           t,
                'z':           z,
                'y':           orig_y,
                'x':           orig_x,
                'phys_z':      z      * SCALE[0],
                'phys_y':      orig_y * SCALE[1],
                'phys_x':      orig_x * SCALE[2],
                'score':       nms_scores[i],
                'parent_phys': None,
            })
            global_nid += 1

        all_nodes_by_t[t] = nodes_t

        if t > 0:
            links = link_timepoints(all_nodes_by_t[t-1], all_nodes_by_t[t])
            for lk in links:
                edges_list.append((lk['source_id'], lk['target_id']))
                for n in all_nodes_by_t[t]:
                    if n['id'] == lk['target_id']:
                        n['parent_phys'] = lk['source_phys']
                        break

        if t % 20 == 0 or t == n_t - 1:
            print(f'  t={t:3d}/{n_t-1} | {len(nuclei)} nuclei detected')

    # short track filter via NetworkX connected components
    G = nx.Graph()
    for t_nodes in all_nodes_by_t.values():
        for n in t_nodes:
            G.add_node(n['id'])
    for src, tgt in edges_list:
        G.add_edge(src, tgt)

    keep = set()
    for comp in nx.connected_components(G):
        if len(comp) >= MIN_TRACK_LEN:
            keep.update(comp)

    # node rows
    n_nodes_kept = 0
    for t_nodes in all_nodes_by_t.values():
        for n in t_nodes:
            if n['id'] not in keep:
                continue
            all_rows.append({
                'dataset':   folder_name,
                'row_type':  'node',
                'node_id':   n['id'],
                't':         n['t'],
                'z':         int(round(n['z'])),
                'y':         int(round(n['y'])),
                'x':         int(round(n['x'])),
                'source_id': -1,
                'target_id': -1,
            })
            n_nodes_kept += 1

    # edge rows
    n_edges_kept = 0
    for src, tgt in edges_list:
        if src in keep and tgt in keep:
            all_rows.append({
                'dataset':   folder_name,
                'row_type':  'edge',
                'node_id':   -1,
                't':         -1,
                'z':         -1,
                'y':         -1,
                'x':         -1,
                'source_id': src,
                'target_id': tgt,
            })
            n_edges_kept += 1

    print(f'  -> kept: {n_nodes_kept} nodes | {n_edges_kept} edges '
          f'(min_track_len={MIN_TRACK_LEN})')

# write submission
submission = pd.DataFrame(
    all_rows,
    columns=['dataset','row_type','node_id','t','z','y','x','source_id','target_id']
)
submission.index.name = 'id'
sub_path = OUT_DIR / 'submission.csv'
submission.to_csv(sub_path)

print(f'\n{"="*55}')
print(f'submission.csv: {sub_path}')
print(f'  Total rows : {len(submission):,}')
print(f'  Node rows  : {(submission.row_type=="node").sum():,}')
print(f'  Edge rows  : {(submission.row_type=="edge").sum():,}')
print(f'  Datasets   : {submission.dataset.nunique()}')

# format checks
assert set(submission.columns) == {
    'dataset','row_type','node_id','t','z','y','x','source_id','target_id'}
assert submission.index.name == 'id'
assert submission[submission.row_type=='node']['source_id'].eq(-1).all()
assert submission[submission.row_type=='node']['target_id'].eq(-1).all()
assert submission[submission.row_type=='edge']['node_id'].eq(-1).all()
print('Format checks passed.')
print(f'{"="*55}')

Test datasets: 4
  44b6_0113de3b
  44b6_0b24845f
  6bba_05b6850b
  6bba_05db0fb1

44b6_0113de3b | 100 timepoints | shape (100, 64, 256, 256)
  t=  0/99 | 614 nuclei detected
  t= 20/99 | 582 nuclei detected
  t= 40/99 | 602 nuclei detected


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ds in submission.dataset.unique():
    nd = submission[(submission.dataset==ds) & (submission.row_type=='node')]
    counts = nd.groupby('t').size()
    axes[0].plot(counts.index, counts.values, label=ds, alpha=0.8)
axes[0].set_xlabel('Timepoint'); axes[0].set_ylabel('Nodes')
axes[0].set_title('Nodes per timepoint'); axes[0].legend(fontsize=7)

nc = submission[submission.row_type=='node'].groupby(['dataset','t']).size()
axes[1].hist(nc.values, bins=30)
axes[1].set_xlabel('Nodes per timepoint'); axes[1].set_title('Node count distribution')

ec = submission[submission.row_type=='edge'].groupby('dataset').size()
axes[2].bar(range(len(ec)), ec.values)
axes[2].set_xticks(range(len(ec)))
axes[2].set_xticklabels(ec.index, rotation=20, ha='right', fontsize=7)
axes[2].set_title('Edges per dataset')

plt.suptitle(f'Submission summary | {len(submission):,} total rows', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'submission_stats.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Stats plot saved: {OUT_DIR}/submission_stats.png')